# CME Bars Loader - Comprehensive Analysis

**Date**: 2025-10-15  
**Version**: 2.0  

基于CME Bars Loader API，探索不同bar采样方法的特性和footprint数据分析。

**研究目标**:
1. 演示CME Bars Loader的完整API使用
2. 对比不同bar类型（TIME, VOLUME, TICK, DOLLAR）的特性
3. 深度分析footprint数据（POC, delta, volume profile）
4. 统计特性分析（分布、波动率、自相关）
5. 可视化最佳实践
6. 性能优化和缓存策略

**Key Features**:
- 基于mlfinlab的专业实现
- 两级缓存加速（tick + result）
- 完整的footprint数据支持
- 现代化Plotly可视化

## 1. Setup & Import

In [ ]:
# Core imports
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# CME Tick Loader imports
from cme_tick_loader import CMEBarsLoader, FootprintVisualizer, FootprintConfig, ChartAPI

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.precision', 4)

print("✓ All imports successful")

## 2. API Overview - Quick Start

CME Bars Loader提供简洁的API来加载和聚合CME期货数据。

In [ ]:
# 初始化loader（自动检测数据路径）
loader = CMEBarsLoader()

print("✓ CMEBarsLoader initialized")
print(f"  Base path: {loader.base_path}")
print(f"  Tick cache: {loader.tick_cache_dir}")
print(f"  Result cache: {loader.result_cache_dir}")

### Basic Usage Example

In [ ]:
# Load 5-minute TIME bars for GC (Gold)
result = loader.load_bars(
    symbol='GC',
    date='20210104',
    resolution='MIN',
    num_units=5,
    use_cache=True,
    verbose=False
)

# Access data
bars = result['bars']
footprint = result['footprint']

print(f"✓ Loaded {len(bars)} bars")
print(f"✓ Footprint: {len(footprint)} price levels across {len(bars)} bars")
print(f"\nBars columns: {list(bars.columns)}")
print(f"\nFootprint columns: {list(footprint.columns)}")

In [ ]:
# Preview bars
print("First 5 bars:")
bars.head()[['date_time', 'open', 'high', 'low', 'close', 'volume']]

In [ ]:
# Preview footprint (first bar)
first_bar_time = bars['date_time'].iloc[0]
print(f"Footprint for bar at {first_bar_time}:")
footprint.loc[first_bar_time].head(10)

## 3. Load Single Day Data

配置研究参数并加载单日数据。

In [ ]:
# Research configuration
SYMBOL = 'GC'
DATE = '20210104'
TICKSIZE = loader.get_ticksize(SYMBOL)  # 使用API获取ticksize

print(f"Research Configuration:")
print(f"  Symbol: {SYMBOL}")
print(f"  Date: {DATE}")
print(f"  Ticksize: {TICKSIZE}")

## 4. Different Bar Types Comparison

创建4种不同的bar类型并对比它们的特性。

In [ ]:
# Dictionary to store results
results_dict = {}

# 1. TIME bars (5-minute)
print('Creating TIME Bars (5-min)...')
time_result = loader.load_bars(
    symbol=SYMBOL,
    date=DATE,
    resolution='MIN',
    num_units=5,
    use_cache=True,
    verbose=False
)
results_dict['TIME_5min'] = time_result
print(f"  → {len(time_result['bars'])} bars")

# 2. VOLUME bars
print('Creating VOLUME Bars (threshold=10000)...')
volume_result = loader.load_bars(
    symbol=SYMBOL,
    date=DATE,
    resolution='VOLUME',
    num_units=10000,
    use_cache=True,
    verbose=False
)
results_dict['VOLUME_10k'] = volume_result
print(f"  → {len(volume_result['bars'])} bars")

# 3. TICK bars
print('Creating TICK Bars (threshold=1000)...')
tick_result = loader.load_bars(
    symbol=SYMBOL,
    date=DATE,
    resolution='TICK',
    num_units=1000,
    use_cache=True,
    verbose=False
)
results_dict['TICK_1k'] = tick_result
print(f"  → {len(tick_result['bars'])} bars")

# 4. DOLLAR bars
print('Creating DOLLAR Bars (threshold=1000000)...')
dollar_result = loader.load_bars(
    symbol=SYMBOL,
    date=DATE,
    resolution='DOLLAR',
    num_units=1000000,
    use_cache=True,
    verbose=False
)
results_dict['DOLLAR_1M'] = dollar_result
print(f"  → {len(dollar_result['bars'])} bars")

print("\n✓ All bar types created")

### Bar Count Summary

In [ ]:
# Create summary table
summary_data = []
for bar_type, result in results_dict.items():
    bars = result['bars']
    summary_data.append({
        'Bar Type': bar_type,
        'Num Bars': len(bars),
        'Avg Volume': bars['volume'].mean(),
        'Total Volume': bars['volume'].sum(),
        'Price Range': f"{bars['low'].min():.1f} - {bars['high'].max():.1f}"
    })

summary_df = pd.DataFrame(summary_data)
print("Bar Types Summary:")
summary_df

In [ ]:
# Visualize bar counts
fig = go.Figure(data=[
    go.Bar(
        x=summary_df['Bar Type'],
        y=summary_df['Num Bars'],
        text=summary_df['Num Bars'],
        textposition='auto',
        marker_color='steelblue'
    )
])

fig.update_layout(
    title=f'Bar Count Comparison - {SYMBOL} {DATE}',
    xaxis_title='Bar Type',
    yaxis_title='Number of Bars',
    template='plotly_white',
    height=400
)

fig.show()

print("\n观察: TIME bars固定采样，信息驱动的bars（VOLUME/TICK/DOLLAR）在活跃时段更密集")

## 5. Footprint Analysis

深度分析footprint数据：POC (Point of Control)、Delta、Volume Profile等。

### 5.1 Single Bar Footprint Analysis

In [ ]:
# Analyze a specific bar from TIME bars
time_bars = results_dict['TIME_5min']['bars']
time_footprint = results_dict['TIME_5min']['footprint']

# Select a bar with high volume (interesting bar)
high_vol_idx = time_bars['volume'].idxmax()
selected_bar = time_bars.loc[high_vol_idx]
selected_time = selected_bar['date_time']

print(f"Selected Bar (highest volume):")
print(f"  Time: {selected_time}")
print(f"  OHLC: {selected_bar['open']:.1f} / {selected_bar['high']:.1f} / {selected_bar['low']:.1f} / {selected_bar['close']:.1f}")
print(f"  Volume: {selected_bar['volume']:.0f}")

# Extract footprint for this bar
bar_footprint = time_footprint.loc[selected_time]

print(f"\nFootprint details:")
print(f"  Price levels: {len(bar_footprint)}")
print(f"  Total volume: {bar_footprint['total_vol'].sum():.0f}")
print(f"  Bid volume: {bar_footprint['bid_vol'].sum():.0f}")
print(f"  Ask volume: {bar_footprint['ask_vol'].sum():.0f}")
print(f"  Net delta: {bar_footprint['delta'].sum():.0f}")

In [ ]:
# Find POC (Point of Control) - price with highest volume
poc_price = bar_footprint['total_vol'].idxmax()
poc_volume = bar_footprint.loc[poc_price, 'total_vol']

print(f"POC Analysis:")
print(f"  POC Price: {poc_price:.1f}")
print(f"  POC Volume: {poc_volume:.0f}")
print(f"  POC Bid/Ask: {bar_footprint.loc[poc_price, 'bid_vol']:.0f} / {bar_footprint.loc[poc_price, 'ask_vol']:.0f}")
print(f"  POC Delta: {bar_footprint.loc[poc_price, 'delta']:.0f}")

# OHLC prices
open_price = bar_footprint[bar_footprint['is_open']].index[0] if bar_footprint['is_open'].any() else None
high_price = bar_footprint[bar_footprint['is_high']].index[0] if bar_footprint['is_high'].any() else None
low_price = bar_footprint[bar_footprint['is_low']].index[0] if bar_footprint['is_low'].any() else None
close_price = bar_footprint[bar_footprint['is_close']].index[0] if bar_footprint['is_close'].any() else None

print(f"\nOHLC from footprint:")
print(f"  Open: {open_price}")
print(f"  High: {high_price}")
print(f"  Low: {low_price}")
print(f"  Close: {close_price}")

In [ ]:
# Visualize Volume Profile for this bar
fig = go.Figure()

# Volume bars
fig.add_trace(go.Bar(
    y=bar_footprint.index,
    x=bar_footprint['total_vol'],
    orientation='h',
    name='Total Volume',
    marker_color='lightblue',
    opacity=0.7
))

# Add POC line
fig.add_hline(y=poc_price, line_dash="dash", line_color="red", 
              annotation_text=f"POC: {poc_price:.1f}", annotation_position="right")

# Add OHLC markers
if open_price:
    fig.add_hline(y=open_price, line_dash="dot", line_color="green", 
                  annotation_text=f"O: {open_price:.1f}", annotation_position="left")
if high_price:
    fig.add_hline(y=high_price, line_dash="dot", line_color="darkgreen", 
                  annotation_text=f"H: {high_price:.1f}", annotation_position="left")
if low_price:
    fig.add_hline(y=low_price, line_dash="dot", line_color="darkred", 
                  annotation_text=f"L: {low_price:.1f}", annotation_position="left")
if close_price:
    fig.add_hline(y=close_price, line_dash="dot", line_color="blue", 
                  annotation_text=f"C: {close_price:.1f}", annotation_position="left")

fig.update_layout(
    title=f'Volume Profile - Bar at {selected_time}',
    xaxis_title='Volume',
    yaxis_title='Price',
    template='plotly_white',
    height=600,
    showlegend=True
)

fig.show()

### 5.2 Delta Analysis

In [ ]:
# Visualize Delta distribution for the selected bar
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Bid vs Ask Volume', 'Delta by Price Level'),
    horizontal_spacing=0.12
)

# Subplot 1: Bid vs Ask
fig.add_trace(
    go.Bar(y=bar_footprint.index, x=bar_footprint['bid_vol'], 
           orientation='h', name='Bid', marker_color='red', opacity=0.7),
    row=1, col=1
)
fig.add_trace(
    go.Bar(y=bar_footprint.index, x=bar_footprint['ask_vol'], 
           orientation='h', name='Ask', marker_color='green', opacity=0.7),
    row=1, col=1
)

# Subplot 2: Delta
colors = ['green' if d > 0 else 'red' for d in bar_footprint['delta']]
fig.add_trace(
    go.Bar(y=bar_footprint.index, x=bar_footprint['delta'], 
           orientation='h', name='Delta', marker_color=colors, opacity=0.7),
    row=1, col=2
)

fig.update_xaxes(title_text="Volume", row=1, col=1)
fig.update_xaxes(title_text="Delta (Ask - Bid)", row=1, col=2)
fig.update_yaxes(title_text="Price", row=1, col=1)
fig.update_yaxes(title_text="Price", row=1, col=2)

fig.update_layout(
    title_text=f'Bid/Ask/Delta Analysis - Bar at {selected_time}',
    template='plotly_white',
    height=600,
    showlegend=True
)

fig.show()

print(f"\n解读:")
print(f"  Net Delta: {bar_footprint['delta'].sum():.0f} (正值=买方主导，负值=卖方主导)")
print(f"  Delta Range: {bar_footprint['delta'].min():.0f} to {bar_footprint['delta'].max():.0f}")

### 5.3 Cumulative Delta Across All Bars

In [ ]:
# Calculate cumulative delta for each bar
bar_deltas = []
for timestamp in time_footprint.index.get_level_values(0).unique():
    bar_fp = time_footprint.loc[timestamp]
    bar_deltas.append({
        'timestamp': timestamp,
        'net_delta': bar_fp['delta'].sum(),
        'total_volume': bar_fp['total_vol'].sum()
    })

delta_df = pd.DataFrame(bar_deltas)
delta_df['cumulative_delta'] = delta_df['net_delta'].cumsum()

# Visualize using ChartAPI
chart = ChartAPI('GC', tick_size=TICKSIZE)
chart.create_candlestick(
    time_bars,
    footprint_df=time_footprint,
    show_volume=True,
    show_cum_delta=True
)
chart.show()

print(f"\nCumulative Delta Summary:")
print(f"  Final Cumulative Delta: {delta_df['cumulative_delta'].iloc[-1]:.0f}")
print(f"  Max Cumulative Delta: {delta_df['cumulative_delta'].max():.0f}")
print(f"  Min Cumulative Delta: {delta_df['cumulative_delta'].min():.0f}")

## 6. Statistical Properties

分析不同bar类型的统计特性：收益分布、波动率、自相关等。

### 6.1 Calculate Log Returns

In [ ]:
# Add log returns to all bar types
for bar_type, result in results_dict.items():
    bars = result['bars']
    bars['log_ret'] = np.log(bars['close']).diff().fillna(0)
    bars['abs_ret'] = bars['log_ret'].abs()

print("✓ Log returns calculated for all bar types")

### 6.2 Return Distribution Comparison

In [ ]:
# Create distribution plots
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f'{bt} Returns' for bt in results_dict.keys()],
    vertical_spacing=0.12,
    horizontal_spacing=0.1
)

positions = [(1,1), (1,2), (2,1), (2,2)]
for (bar_type, result), (row, col) in zip(results_dict.items(), positions):
    returns = result['bars']['log_ret'].dropna()
    
    fig.add_trace(
        go.Histogram(x=returns, nbinsx=50, name=bar_type, opacity=0.7),
        row=row, col=col
    )
    
    # Add normal distribution overlay
    mu, sigma = returns.mean(), returns.std()
    x_range = np.linspace(returns.min(), returns.max(), 100)
    normal_dist = (1/(sigma * np.sqrt(2*np.pi))) * np.exp(-0.5*((x_range - mu)/sigma)**2)
    normal_dist = normal_dist * len(returns) * (returns.max() - returns.min()) / 50
    
    fig.add_trace(
        go.Scatter(x=x_range, y=normal_dist, mode='lines', 
                   line=dict(color='red', dash='dash'), name='Normal'),
        row=row, col=col
    )

fig.update_layout(
    title_text='Log Return Distributions (with Normal overlay)',
    template='plotly_white',
    height=700,
    showlegend=False
)

fig.show()

# Statistical summary
print("\nReturn Statistics:")
stats_data = []
for bar_type, result in results_dict.items():
    returns = result['bars']['log_ret'].dropna()
    stats_data.append({
        'Bar Type': bar_type,
        'Mean': returns.mean(),
        'Std': returns.std(),
        'Skew': returns.skew(),
        'Kurtosis': returns.kurtosis()
    })

stats_df = pd.DataFrame(stats_data)
stats_df

### 6.3 Volatility Analysis

In [ ]:
# Calculate three types of volatility estimators
volatility_data = []

for bar_type, result in results_dict.items():
    bars = result['bars']
    returns = bars['log_ret'].dropna()
    
    # 1. Close-to-close (standard method)
    vol_cc = returns.std()
    
    # 2. Parkinson (high-low based)
    hl_ratio = np.log(bars['high'] / bars['low'])
    vol_parkinson = np.sqrt((1 / (4 * np.log(2))) * (hl_ratio ** 2).mean())
    
    # 3. Garman-Klass (OHLC based)
    hl = np.log(bars['high'] / bars['low'])
    co = np.log(bars['close'] / bars['open'])
    vol_gk = np.sqrt(0.5 * (hl ** 2).mean() - (2 * np.log(2) - 1) * (co ** 2).mean())
    
    # Convert to ticks for better interpretation
    avg_price = bars['close'].mean()
    volatility_data.append({
        'Bar Type': bar_type,
        'Close-to-Close': vol_cc,
        'Parkinson': vol_parkinson,
        'Garman-Klass': vol_gk,
        'CC (ticks)': vol_cc * avg_price / TICKSIZE,
        'Parkinson (ticks)': vol_parkinson * avg_price / TICKSIZE,
        'GK (ticks)': vol_gk * avg_price / TICKSIZE
    })

vol_df = pd.DataFrame(volatility_data)
print("Volatility Comparison:")
print("\n说明:")
print("  - Close-to-Close: 只用收盘价，最简单")
print("  - Parkinson: 用high-low range，效率高5倍")
print("  - Garman-Klass: 用OHLC，效率高7.4倍")
print("  - Ticks: 转换为tick单位更直观\n")
vol_df

In [ ]:
# Visualize volatility in ticks
fig = go.Figure()

x = vol_df['Bar Type']

fig.add_trace(go.Bar(
    x=x, y=vol_df['CC (ticks)'],
    name='Close-to-Close',
    opacity=0.8
))

fig.add_trace(go.Bar(
    x=x, y=vol_df['Parkinson (ticks)'],
    name='Parkinson',
    opacity=0.8
))

fig.add_trace(go.Bar(
    x=x, y=vol_df['GK (ticks)'],
    name='Garman-Klass',
    opacity=0.8
))

fig.update_layout(
    title='Volatility Comparison in Ticks',
    xaxis_title='Bar Type',
    yaxis_title='Volatility (ticks per bar)',
    barmode='group',
    template='plotly_white',
    height=500
)

fig.show()

print("\n解读: 典型的一个bar，价格波动约X个ticks")

### 6.4 Autocorrelation Analysis

In [ ]:
# Calculate ACF for each bar type
from pandas.plotting import autocorrelation_plot

def calculate_acf(series, nlags=20):
    """Calculate autocorrelation function"""
    acf_values = []
    for lag in range(nlags + 1):
        if lag == 0:
            acf_values.append(1.0)
        else:
            acf_values.append(series.autocorr(lag=lag))
    return acf_values

# Plot ACF comparison
fig = go.Figure()

nlags = 20
for bar_type, result in results_dict.items():
    returns = result['bars']['log_ret'].dropna()
    acf_values = calculate_acf(returns, nlags)
    
    fig.add_trace(go.Scatter(
        x=list(range(nlags + 1)),
        y=acf_values,
        mode='lines+markers',
        name=bar_type,
        line=dict(width=2),
        marker=dict(size=6)
    ))

# Add confidence interval
n = len(results_dict['TIME_5min']['bars'])
conf_interval = 1.96 / np.sqrt(n)
fig.add_hline(y=conf_interval, line_dash="dash", line_color="red", opacity=0.5)
fig.add_hline(y=-conf_interval, line_dash="dash", line_color="red", opacity=0.5)
fig.add_hline(y=0, line_dash="dot", line_color="gray")

fig.update_layout(
    title='Autocorrelation Function (ACF) Comparison',
    xaxis_title='Lag',
    yaxis_title='ACF',
    template='plotly_white',
    height=500,
    hovermode='x unified'
)

fig.show()

print("\n观察:")
print("  - 信息驱动的bars（DOLLAR/VOLUME）通常有更低的ACF")
print("  - 更接近IID (Independent and Identically Distributed)")
print("  - 红色虚线: 95%置信区间")

## 7. Visualization Examples

使用FootprintVisualizer进行专业的footprint可视化。

In [ ]:
# Initialize visualizer
viz = FootprintVisualizer()

# Select a subset of bars for visualization (last 20 bars)
time_footprint = results_dict['TIME_5min']['footprint']
timestamps = time_footprint.index.get_level_values(0).unique()[-20:]
subset_footprint = time_footprint.loc[timestamps]

print(f"Visualizing last {len(timestamps)} bars")
print(f"Time range: {timestamps[0]} to {timestamps[-1]}")

In [ ]:
# DEPRECATED: FootprintVisualizer.plot_footprint() has known issues
# Use ChartAPI instead for professional charts
print("⚠️  FootprintVisualizer.plot_footprint() is deprecated due to known issues.")
print("   Please use ChartAPI for candlestick charts with volume and cumulative delta.")
print("   See Cell 24 for ChartAPI usage example.")

# Example (not executed):
# chart = ChartAPI('GC', tick_size=TICKSIZE)
# chart.create_candlestick(time_bars, footprint_df=time_footprint,
#                          show_volume=True, show_cum_delta=True)
# chart.show()

In [ ]:
# Use ChartAPI for professional candlestick chart
chart = ChartAPI(
    symbol='GC',
    tick_size=TICKSIZE
)

# Select last 20 bars
timestamps = time_footprint.index.get_level_values(0).unique()[-20:]
subset_bars = time_bars.loc[time_bars['date_time'].isin(timestamps)]
subset_footprint = time_footprint.loc[timestamps]

chart.create_candlestick(
    subset_bars,
    footprint_df=subset_footprint,
    show_volume=True,
    show_cum_delta=True,
    show_atr=False
)

fig = chart.get_figure()
fig.update_layout(title=f"{SYMBOL} {DATE} - Last 20 Bars (5-min) with Volume & Cumulative Delta")
fig.show()

print("\nChart Features:")
print("  - Main chart: Candlestick (OHLC)")
print("  - Subplot 1: Volume bars")
print("  - Subplot 2: Cumulative Delta (buying/selling pressure)")

## 8. Multi-Day Analysis

分析多日数据，观察跨日模式。

In [ ]:
# Load 3-day range
multi_day_result = loader.load_date_range(
    symbol=SYMBOL,
    start_date='20210104',
    end_date='20210106',
    resolution='MIN',
    num_units=5,
    use_cache=True,
    verbose=False
)

multi_bars = multi_day_result['bars']
print(f"✓ Loaded {len(multi_bars)} bars across 3 days")
print(f"  Date range: {multi_bars['date_time'].min()} to {multi_bars['date_time'].max()}")

In [ ]:
# Calculate daily statistics
multi_bars['date'] = multi_bars['date_time'].dt.date
multi_bars['log_ret'] = np.log(multi_bars['close']).diff().fillna(0)

daily_stats = multi_bars.groupby('date').agg({
    'close': ['first', 'last', 'min', 'max'],
    'volume': ['sum', 'mean'],
    'log_ret': ['std', 'mean']
}).round(4)

daily_stats.columns = ['_'.join(col) for col in daily_stats.columns]
print("\nDaily Statistics:")
daily_stats

In [ ]:
# Visualize multi-day price action using ChartAPI
chart = ChartAPI('GC', tick_size=TICKSIZE)
chart.create_candlestick(
    multi_bars,
    footprint_df=multi_day_result['footprint'],
    show_volume=True,
    show_cum_delta=False,  # Optional: can enable if needed
    show_atr=False
)

fig = chart.get_figure()
fig.update_layout(
    title=f'{SYMBOL} - 3 Day Analysis (5-min bars)',
    height=700
)
fig.show()

## 9. Cache Performance Analysis

测试缓存系统的性能提升。

In [ ]:
import time

# Test 1: Load with cache (warm)
start = time.time()
result_cached = loader.load_bars(
    symbol=SYMBOL,
    date=DATE,
    resolution='MIN',
    num_units=5,
    use_cache=True,
    refresh_cache=False,
    verbose=False
)
time_cached = time.time() - start

# Test 2: Load without cache (cold)
start = time.time()
result_nocache = loader.load_bars(
    symbol=SYMBOL,
    date=DATE,
    resolution='MIN',
    num_units=5,
    use_cache=False,
    verbose=False
)
time_nocache = time.time() - start

print("Cache Performance Test:")
print(f"  With cache: {time_cached:.4f}s")
print(f"  Without cache: {time_nocache:.4f}s")
print(f"  Speedup: {time_nocache/time_cached:.2f}x")

# Cache info
cache_info = loader.get_cache_info()
print(f"\nCache Statistics:")
print(f"  Tick cache: {cache_info['tick_cache']['count']} files, {cache_info['tick_cache']['size_mb']:.2f} MB")
print(f"  Result cache: {cache_info['result_cache']['count']} files, {cache_info['result_cache']['size_mb']:.2f} MB")
print(f"  Total: {cache_info['total_size_mb']:.2f} MB")

## 10. Advanced Analysis - Market Microstructure Insights

深度分析市场微观结构。

### 10.1 Volume-Volatility Relationship

In [ ]:
# Analyze volume-volatility relationship for TIME bars
time_bars = results_dict['TIME_5min']['bars'].copy()

# Calculate realized volatility (high-low range)
time_bars['hl_range'] = time_bars['high'] - time_bars['low']
time_bars['hl_range_pct'] = time_bars['hl_range'] / time_bars['close']

# Scatter plot
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=time_bars['volume'],
    y=time_bars['hl_range'],
    mode='markers',
    marker=dict(
        size=8,
        color=time_bars['hl_range'],
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title="Range")
    ),
    text=[f"Time: {t}<br>Vol: {v:.0f}<br>Range: {r:.2f}" 
          for t, v, r in zip(time_bars['date_time'], time_bars['volume'], time_bars['hl_range'])],
    hovertemplate='%{text}<extra></extra>'
))

# Add trend line
from scipy import stats
slope, intercept, r_value, p_value, std_err = stats.linregress(
    time_bars['volume'], time_bars['hl_range']
)
x_trend = np.array([time_bars['volume'].min(), time_bars['volume'].max()])
y_trend = slope * x_trend + intercept

fig.add_trace(go.Scatter(
    x=x_trend,
    y=y_trend,
    mode='lines',
    line=dict(color='red', dash='dash', width=2),
    name=f'Trend (R²={r_value**2:.3f})'
))

fig.update_layout(
    title='Volume-Volatility Relationship',
    xaxis_title='Volume',
    yaxis_title='High-Low Range',
    template='plotly_white',
    height=500
)

fig.show()

print(f"\nCorrelation Analysis:")
print(f"  Pearson R: {r_value:.4f}")
print(f"  R-squared: {r_value**2:.4f}")
print(f"  P-value: {p_value:.6f}")
print(f"\n解读: {'显著正相关' if p_value < 0.05 and r_value > 0 else '相关性不显著'}")

### 10.2 Trade Aggressiveness (Delta Imbalance)

In [ ]:
# Calculate trade aggressiveness for each bar
time_footprint = results_dict['TIME_5min']['footprint']

bar_aggression = []
for timestamp in time_footprint.index.get_level_values(0).unique():
    bar_fp = time_footprint.loc[timestamp]
    
    total_vol = bar_fp['total_vol'].sum()
    net_delta = bar_fp['delta'].sum()
    
    # Aggressiveness ratio: |delta| / total_volume
    # High ratio = imbalanced (aggressive buying or selling)
    # Low ratio = balanced (neutral)
    aggression = abs(net_delta) / total_vol if total_vol > 0 else 0
    
    bar_aggression.append({
        'timestamp': timestamp,
        'net_delta': net_delta,
        'total_volume': total_vol,
        'aggression_ratio': aggression,
        'direction': 'BUY' if net_delta > 0 else 'SELL'
    })

aggression_df = pd.DataFrame(bar_aggression)

# Visualize
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Net Delta', 'Aggressiveness Ratio'),
    vertical_spacing=0.12
)

# Net delta
colors = ['green' if d > 0 else 'red' for d in aggression_df['net_delta']]
fig.add_trace(
    go.Bar(x=aggression_df.index, y=aggression_df['net_delta'], 
           marker_color=colors, opacity=0.7),
    row=1, col=1
)

# Aggressiveness ratio
fig.add_trace(
    go.Scatter(x=aggression_df.index, y=aggression_df['aggression_ratio'],
               mode='lines+markers', line=dict(color='blue', width=2),
               marker=dict(size=4)),
    row=2, col=1
)

# Add threshold line (e.g., 0.3 = 30% imbalance)
fig.add_hline(y=0.3, line_dash="dash", line_color="red", 
              annotation_text="High Aggression", row=2, col=1)

fig.update_xaxes(title_text="Bar Index", row=2, col=1)
fig.update_yaxes(title_text="Net Delta", row=1, col=1)
fig.update_yaxes(title_text="Aggression Ratio", row=2, col=1)

fig.update_layout(
    title_text='Trade Aggressiveness Analysis',
    template='plotly_white',
    height=700,
    showlegend=False
)

fig.show()

print(f"\nAggressiveness Statistics:")
print(f"  Mean ratio: {aggression_df['aggression_ratio'].mean():.4f}")
print(f"  Max ratio: {aggression_df['aggression_ratio'].max():.4f}")
print(f"  Bars > 30% imbalance: {(aggression_df['aggression_ratio'] > 0.3).sum()}")
print(f"\n解读: Ratio > 0.3 表示该bar交易不平衡，可能有知情交易者")

### 10.3 Price Impact of Volume

In [ ]:
# Analyze how volume impacts subsequent price changes
time_bars = results_dict['TIME_5min']['bars'].copy()

# Calculate forward returns (1-bar ahead)
time_bars['fwd_ret'] = time_bars['log_ret'].shift(-1)

# Categorize volume into quintiles
time_bars['volume_quintile'] = pd.qcut(time_bars['volume'], q=5, labels=['Q1', 'Q2', 'Q3', 'Q4', 'Q5'])

# Calculate average forward return for each quintile
impact_analysis = time_bars.groupby('volume_quintile').agg({
    'fwd_ret': ['mean', 'std', 'count'],
    'volume': 'mean'
}).round(6)

impact_analysis.columns = ['_'.join(col) for col in impact_analysis.columns]
print("Price Impact Analysis:")
print(impact_analysis)
print("\n解读: 观察高成交量quintile是否导致更大的价格变化")

## 11. Summary & Key Insights

总结本次分析的主要发现。

In [ ]:
print("="*80)
print("CME Bars Loader Analysis - Key Insights")
print("="*80)

print("\n1. API使用")
print("   ✓ 简洁的API: load_bars()一行代码完成所有聚合")
print("   ✓ 支持4种bar类型: TIME, VOLUME, TICK, DOLLAR")
print("   ✓ 完整的footprint数据: bid/ask volume, delta, OHLC flags")
print("   ✓ 两级缓存: 显著提升性能")

print("\n2. Bar类型对比")
print(f"   - TIME bars: {len(results_dict['TIME_5min']['bars'])} bars (固定时间采样)")
print(f"   - VOLUME bars: {len(results_dict['VOLUME_10k']['bars'])} bars (活跃时段更密集)")
print(f"   - TICK bars: {len(results_dict['TICK_1k']['bars'])} bars")
print(f"   - DOLLAR bars: {len(results_dict['DOLLAR_1M']['bars'])} bars")

print("\n3. 统计特性")
print("   - 信息驱动的bars (DOLLAR/VOLUME) 有更低的自相关")
print("   - 更接近IID假设，适合统计建模")
print("   - 波动率估计: Garman-Klass效率最高")

print("\n4. Footprint分析")
print("   - POC (Point of Control): 最大成交量价位")
print("   - Delta: 买卖不平衡指标")
print("   - Cumulative Delta: 追踪市场情绪变化")
print("   - Aggressiveness Ratio: 识别知情交易")

print("\n5. 性能优化")
print(f"   - 缓存加速: {time_nocache/time_cached:.2f}x")
print(f"   - 缓存大小: {cache_info['total_size_mb']:.2f} MB")
print("   - 按天管理: 避免跨天状态污染")

print("\n6. 实用工具")
print("   ✓ FootprintVisualizer: 专业的ATAS风格可视化")
print("   ✓ load_date_range(): 便捷的多日数据加载")
print("   ✓ 完整的统计分析工具")

print("\n" + "="*80)
print("下一步研究方向:")
print("  1. 基于delta构建交易信号")
print("  2. 多合约footprint对比分析")
print("  3. 实时流式数据处理")
print("  4. 机器学习特征工程")
print("="*80)

## Appendix: Quick Reference

### Common Operations

```python
# Initialize
loader = CMEBarsLoader()

# Load single day
result = loader.load_bars(
    symbol='GC',
    date='20210104',
    resolution='MIN',
    num_units=5,
)

# Load date range
result = loader.load_date_range(
    symbol='GC',
    start_date='20210104',
    end_date='20210106',
    resolution='MIN',
    num_units=5
)

# Access data
bars = result['bars']
footprint = result['footprint']

# Visualize
viz = FootprintVisualizer()
fig = viz.plot_footprint(footprint, ticksize=0.1)
fig.show()

# Cache management
loader.get_cache_info()
loader.clear_cache(level='result')
```